## Sample project with CrewAI
- to create a sample resume checking application
- Read the pdf resume file and check it with a JD
- find the revelvence of the resume agains the job description
- re-write the resmue based on the job description to pass the ATS

In [4]:
%pip install crewai | tail -n 1
%pip install crewai-tools | tail -n 1
%pip install litellm | tail -n 1
%pip install docling| tail -n 1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-chroma 1.1.0 requires chromadb<2.0.0,>=1.3.5, but you have chromadb 1.1.1 which is incompatible.
litellm 1.83.10 requires pydantic==2.12.5, but you have pydantic 2.11.10 which is incompatible.
litellm 1.83.10 requires python-dotenv==1.0.1, but you have python-dotenv 1.1.1 which is incompatible.
crewai-tools 1.14.2 requires tiktoken~=0.8.0, but you have tiktoken 0.12.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.83.10 requires pydantic==2.12.5, but you have pydantic 2.11.10 which is incompatible.
litellm 1.83.10 requires python-dotenv==1.0.1, but you have python-dotenv 1.1.1 whic

## Tools
- a duck duck go search tool

In [15]:
from ddgs import DDGS
from crewai.tools import tool

@tool("search_tool")
def search_tool(query, max_results=3):
    """tool to search web"""
    with DDGS() as search:
        results = [r for r in 
        search.text(
            query, 
            max_results=max_results,
            safe_search=True
        )]
    
    return results

# to test the tool
search_tool.run("who was the fist president of USA?")


[{'title': 'List of presidents of the United States - Wikipedia',
  'href': 'https://en.wikipedia.org/wiki/List_of_Presidents_of_the_United_States',
  'body': 'John Tyler was the first vicepresidentto assume the presidency during a presidential term, setting the precedent that a vicepresidentwhodoes so becomes the fully functioningpresidentwith a new, distinct administration. [13] Throughout most of its history, American politics has been dominated by political parties.'},
 {'title': 'list of presidents of the United States - Encyclopedia Britannica',
  'href': 'https://www.britannica.com/topic/Presidents-of-the-United-States-1846696',
  'body': 'As the head of the government of the United States, thepresidentis arguably the most powerful government official in the world. Thepresidentis elected to a four-year term via an electoral college system. Since the Twenty-second Amendment was adopted in 1951, the American presidency has been'},
 {'title': "Before the White House: Who Really Was

## LLM
- need to use a Groq model in Crewai

In [ ]:
import os
from crewai import LLM
import dotenv

dotenv.load_dotenv("../.env")

os.environ["GROQ_API_KEY"] = os.getenv("GROQ")

llm = LLM(
    model="groq/llama-3.3-70b-versatile",
    temperature=0
)

#llm call should be a list of dictionaries
# dictionary content = role and message
llm.call([{
    "role": "user",
    "content": "which is the capital of Hessen, Germany"
}])

'The capital of Hessen, Germany is Wiesbaden.'

In [17]:
# Resume and JD

# need to input a resume (pdf)
# convert the resume to markdown
import os
from docling.document_converter import DocumentConverter

resume_path = "../data/resume.pdf"

def convert_to_markdown(doc_path):
    if not os.path.exists(doc_path):
        raise FileNotFoundError(f"Document not found at path: {doc_path}")
    converter = DocumentConverter()
    markdown = converter.convert(doc_path).document.export_to_markdown()
    return markdown

markdown_resume = convert_to_markdown(resume_path)
# print(markdown_resume)

# JD
jd_path = "../data/jd.txt"

def extract_jd(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"JD file not found at path: {path}")
    with open(path, "r") as f:
        jd = f.read()

    return jd

jd = extract_jd(jd_path)
# print(jd)

In [ ]:
from crewai import Agent, Task, Crew, Process

carrier_coach = Agent(
    role = "Senior Carrier Coach",
    goal = """
    Your goal is to analyze the resume and find the key strengths and weakness in the resme
    - Objectively analyze the resume and find the strengths
    - Find the gaps in the resume, and potential improvement avenues
    """,
    backstory = """
    You are a senior hr person, have extensive experience in Tech-recurting
    you have recruted many candidates for top Technology firms like Meta, Google, and Apple
    """,
    llm = llm,
    verbose=True,
    delegation=False,
    tools=[search_tool]
)

resume_check = Task(
    description="Analyze the {resume} and come up with the strenghs and weekness of this resume",
    expected_output="itemized strengths and weekndess",
    agent=carrier_coach
)

resume_writer = Agent(
    role="Senior Resume Writer",
    goal="Your goal is to write ATS friendly tech resumes",
    backstory="""You have worked in Multiple currier coatching firms and recruitment firms as a
    resume wirter and has 10 years of experience as tech resume witer and has good grasp of technology""",
    llm=llm,
    delegation=True,
    verbose=True
)

resume_witer_task = Task(
    description="Write engaging, goal centric resume bullet and points based on the suggestions provided by the carrier coatch",
    expected_output="Bullet and points which is clear and action oriented",
    agent=resume_writer
)

crew = Crew(
    agents=[carrier_coach, resume_writer],
    tasks=[resume_check, resume_witer_task],
    process=Process.sequential,
    verbose=True
)

crew.kickoff({"resume": markdown_resume})


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 695dc78a-308c-4b67-9ffd-b8f2bcccfbc5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the <!-- image -->                                                                               │
│                                                                                                                 │
│  JOB SKILLS                                                                                                     │
│                                                                                                                 │
│  ## SKILLS                                                                                                      │
│                                                                                                                 │
│  ## EXPERIENCE                                                                                                  │
│                                                                                                                 │
│  Mar 2022- Dec 2023                                                                                             │
│                                                                                                                 │
│  Aneesh Cherian K German Residence permit holder                                                                │
│                                                                                                                 │
│  Senior Data Scientist (7+ Years' Experience)                                                                   │
│                                                                                                                 │
│  Address: Sindlingen, Frankfurt am Main                                                                         │
│                                                                                                                 │
│  Phone: +4915511223960                                                                                          │
│                                                                                                                 │
│  Sex: Male | DOB: 02.12.1984 | Nationality: Indian                                                              │
│                                                                                                                 │
│  LinkedIn: https://linkedin.com/in/aneeshcheriank                                                               │
│                                                                                                                 │
│  Github: https://github.com/aneeshcheriank                                                                      │
│                                                                                                                 │
│  - Automated data mapping using an NLP model (Keras &amp; pandas) reduces custodian onboarding time by 50%      │
│  - Optimized deep learning model using pandas, Scikit-Learn, &amp; feedback loops 5% recall gain                │
│  - Built a neural network (Tensoflow &amp; Scikit-learn) for transaction note-to-parent mapping, achieving 98%  │
│  accuracy with a 5% transaction classification improvement                                                      │
│  - Improved dialogue summarization ROUGE-n precision by 10% through selective fine-tuning of pre-trained LLMs   │
│  with Transformers, Pytorch, and PEFT                                                                           │
│  - Adept in AI/ML (Statistics, Machine Learning, Deep Learning, NLP): proven business impact                    │
│  - Built a Minimum Viable Product (RAG) using Langchain

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Carrier Coach                                                                                    │
│                                                                                                                 │
│  Task: Analyze the <!-- image -->                                                                               │
│                                                                                                                 │
│  JOB SKILLS                                                                                                     │
│                                                                                                                 │
│  ## SKILLS                                                                                                      │
│                                                                                                                 │
│  ## EXPERIENCE                                                                                                  │
│                                                                                                                 │
│  Mar 2022- Dec 2023                                                                                             │
│                                                                                                                 │
│  Aneesh Cherian K German Residence permit holder                                                                │
│                                                                                                                 │
│  Senior Data Scientist (7+ Years' Experience)                                                                   │
│                                                                                                                 │
│  Address: Sindlingen, Frankfurt am Main                                                                         │
│                                                                                                                 │
│  Phone: +4915511223960                                                                                          │
│                                                                                                                 │
│  Sex: Male | DOB: 02.12.1984 | Nationality: Indian                                                              │
│                                                                                                                 │
│  LinkedIn: https://linkedin.com/in/aneeshcheriank                                                               │
│                                                                                                                 │
│  Github: https://github.com/aneeshcheriank                                                                      │
│                                                                                                                 │
│  - Automated data mapping using an NLP model (Keras &amp; pandas) reduces custodian onboarding time by 50%      │
│  - Optimized deep learning model using pandas, Scikit-Learn, &amp; feedback loops 5% recall gain                │
│  - Built a neural network (Tensoflow &amp; Scikit-learn) for transaction note-to-parent mapping, achieving 98%  │
│  accuracy with a 5% transaction classification improvement                                                      │
│  - Improved dialogue summarization ROUGE-n precision by 10% through selective fine-tuning of pre-trained LLMs   │
│  with Transformers, Pytorch, and PEFT                                                                           │
│  - Adept in AI/ML (Statistics, Machine Learning, Deep L

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: litellm.BadRequestError: GroqException - {"error":{"message":"tool call validation failed: attempted    │
│  to call tool 'analyze_resume' which was not in                                                                 │
│  request.tools","type":"invalid_request_error","code":"tool_use_failed","failed_generation":"\u003cfunction=an  │
│  alyze_resume\u003e{\"resume\": \"## JOB SKILLS\\n\\n## SKILLS\\n\\n## EXPERIENCE\\n\\nMar 2022- Dec            │
│  2023\\n\\nAneesh Cherian K German Residence permit holder\\n\\nSenior Data Scientist (7+ Years'                │
│  Experience)\\n\\nAddress: Sindlingen, Frankfurt am Main\\n\\nPhone: +4915511223960\\n\\nSex: Male | DOB:       │
│  02.12.1984 | Nationality: Indian\\n\\nLinkedIn: https://linkedin.com/in/aneeshcheriank\\n\\nGithub:            │
│  https://github.com/aneeshcheriank\\n\\n- Automated data mapping using an NLP model (Keras \u0026amp; pandas)   │
│  reduces custodian onboarding time by 50%\\n- Optimized deep learning model using pandas, Scikit-Learn,         │
│  \u0026amp; feedback loops 5% recall gain\\n- Built a neural network (Tensoflow \u0026amp; Scikit-learn) for    │
│  transaction note-to-parent mapping, achieving 98% accuracy with a 5% transaction classification                │
│  improvement\\n- Improved dialogue summarization ROUGE-n precision by 10% through selective fine-tuning of      │
│  pre-trained LLMs with Transformers, Pytorch, and PEFT\\n- Adept in AI/ML (Statistics, Machine Learning, Deep   │
│  Learning, NLP): proven business impact\\n- Built a Minimum Viable Product (RAG) using Langchain and Chroma     │
│  for PDF information retrieval, improving information retrieval efficiency\\n- Deployed custom multi-threaded   │
│  Python packages (Joblib) with unit tests (Pytest) for AI inference in SageMaker, enhancing efficiency and      │
│  scalability\\n\\n| Libraries     | Transformes, Peft, LangChain, TensorFlow, PyTorch, Keras, Scikit-Learn,     │
│  OpenCV, Pandas, Numpy, SciPy, Pytest, R-shiny, dplyr, re, Flask, PyMuPDF, Joblib                               │
│  |\\n|---------------|----------------------------------------------------------------------------------------  │
│  ----|\\n| DSL \u0026 OCR     | SQL, Textract                                                                   │
│  |\\n| Visualization | Seaborn, Matplot lib, Tableau, R-Shiny                                                   │
│  |\\n| IDEs          | PyCharm, VS Code, Jupyter-notebook/Lab, R Studio                                         │
│  |\\n| CI/CD         | GitLab, GitHub, SonarQube                                                                │
│  |\\n| Tools         | Confluence, Jira, Textract                                                               │
│  |\\n| Cloud         | AWS, Sagemaker, Athena, Tesseract, s3                                                    │
│  |\\n\\n## Lead Data Scientist\\n\\n- Reduced document verification effort by 90% by building a Python package  │
│  (TensorFlow,\\nEnvestnet Asset Management Pvt Ltd, Trivandrum, India\\n\\n- Textract, pandas, Scipy, re,       │
│  PyMuPDF) deployed in the AWS SageMaker pipeline\\n- Spearheaded a team of 3 data scientists, automating        │
│  document verification - 3 FTE reduction\\n- Developed a 99% accurate computer vision model (Tensorflow         │
│  \u0026amp; OpenCV) to identify the matching pages for document comparison\\n- Fostered a productive team       │
│  environment through code review, refactoring, and complexity reduction for 4 AI-powered Python packages in     │
│  GitLab\\n- Facilitated collaboration by documenting AI

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Analyze the <!-- image -->                                                                               │
│                                                                                                                 │
│  JOB SKILLS                                                                                                     │
│                                                                                                                 │
│  ## SKILLS                                                                                                      │
│                                                                                                                 │
│  ## EXPERIENCE                                                                                                  │
│                                                                                                                 │
│  Mar 2022- Dec 2023                                                                                             │
│                                                                                                                 │
│  Aneesh Cherian K German Residence permit holder                                                                │
│                                                                                                                 │
│  Senior Data Scientist (7+ Years' Experience)                                                                   │
│                                                                                                                 │
│  Address: Sindlingen, Frankfurt am Main                                                                         │
│                                                                                                                 │
│  Phone: +4915511223960                                                                                          │
│                                                                                                                 │
│  Sex: Male | DOB: 02.12.1984 | Nationality: Indian                                                              │
│                                                                                                                 │
│  LinkedIn: https://linkedin.com/in/aneeshcheriank                                                               │
│                                                                                                                 │
│  Github: https://github.com/aneeshcheriank                                                                      │
│                                                                                                                 │
│  - Automated data mapping using an NLP model (Keras &amp; pandas) reduces custodian onboarding time by 50%      │
│  - Optimized deep learning model using pandas, Scikit-Learn, &amp; feedback loops 5% recall gain                │
│  - Built a neural network (Tensoflow &amp; Scikit-learn) for transaction note-to-parent mapping, achieving 98%  │
│  accuracy with a 5% transaction classification improvement                                                      │
│  - Improved dialogue summarization ROUGE-n precision by 10% through selective fine-tuning of pre-trained LLMs   │
│  with Transformers, Pytorch, and PEFT                                                                           │
│  - Adept in AI/ML (Statistics, Machine Learning, Deep Learning, NLP): proven business impact                    │
│  - Built a Minimum Viable Product (RAG) using Langchain

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 695dc78a-308c-4b67-9ffd-b8f2bcccfbc5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

BadRequestError: litellm.BadRequestError: GroqException - {"error":{"message":"tool call validation failed: attempted to call tool 'analyze_resume' which was not in request.tools","type":"invalid_request_error","code":"tool_use_failed","failed_generation":"\u003cfunction=analyze_resume\u003e{\"resume\": \"## JOB SKILLS\\n\\n## SKILLS\\n\\n## EXPERIENCE\\n\\nMar 2022- Dec 2023\\n\\nAneesh Cherian K German Residence permit holder\\n\\nSenior Data Scientist (7+ Years' Experience)\\n\\nAddress: Sindlingen, Frankfurt am Main\\n\\nPhone: +4915511223960\\n\\nSex: Male | DOB: 02.12.1984 | Nationality: Indian\\n\\nLinkedIn: https://linkedin.com/in/aneeshcheriank\\n\\nGithub: https://github.com/aneeshcheriank\\n\\n- Automated data mapping using an NLP model (Keras \u0026amp; pandas) reduces custodian onboarding time by 50%\\n- Optimized deep learning model using pandas, Scikit-Learn, \u0026amp; feedback loops 5% recall gain\\n- Built a neural network (Tensoflow \u0026amp; Scikit-learn) for trans